<a href="https://colab.research.google.com/github/luanacanoms/cordoba-housing-data/blob/main/extraccion_api_idealista.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import base64
import pandas as pd
import numpy as np
import json

In [ ]:
from google.colab import files
import pandas as pd
import io

print("Por favor, faça o upload dos dois arquivos (sua base e o foo.csv):")
uploaded = files.upload()

nomes = list(uploaded.keys())

nome_base = [n for n in nomes if 'base' in n.lower() or 'idealista' in n.lower()][0]
nome_foo = [n for n in nomes if 'foo' in n.lower()][0]

df_raspado = pd.read_csv(io.BytesIO(uploaded[nome_base]))
df_gabarito = pd.read_csv(io.BytesIO(uploaded[nome_foo]), nrows=0)

print(f"\n¡Éxito! Archivos leídos: '{nome_base}' y '{nome_foo}'")

Por favor, faça o upload dos dois arquivos (sua base e o foo.csv):


Saving base_cordoba_api.csv to base_cordoba_api.csv
Saving foo.csv to foo.csv

Sucesso! Lidos os arquivos: 'base_cordoba_api.csv' e 'foo.csv'


In [ ]:
import requests
import base64
import pandas as pd
import time
from google.colab import files

apikey = '*****'
client_secret = '*****'
credentials = f"{apikey}:{client_secret}"
encoded_credentials = base64.b64encode(credentials.encode()).decode()

token_url = "https://api.idealista.com/oauth/token"
data = {"grant_type": "client_credentials", "scope": "read"}
headers = {"Authorization": f"Basic {encoded_credentials}", "Content-Type": "application/x-www-form-urlencoded;charset=UTF-8"}

response = requests.post(token_url, data=data, headers=headers)
c = response.json().get("access_token") if response.status_code == 200 else None

In [ ]:
def mapear_imovel_api(imovel_json):
    detalhes = imovel_json.get('detailedType', {})
    estacionamento = imovel_json.get('parkingSpace', {})

    dados_padronizados = {
        'id': imovel_json.get('propertyCode'),
        'url': imovel_json.get('url'),
        'operation': imovel_json.get('operation'),
        'state': imovel_json.get('province'),
        'location_id': imovel_json.get('municipality'),

        'ubicacion_latitude': imovel_json.get('latitude'),
        'ubicacion_longitud': imovel_json.get('longitude'),
        'ubicacion_HasHidden': imovel_json.get('showAddress') == False,
        'ubication__administrativeAreaLevel3': imovel_json.get('district'),
        'ubication__administrativeAreaLevel4': imovel_json.get('neighborhood'),

        'commercialDataId': None,
        'price_raw': imovel_json.get('price'),

        'housetype': imovel_json.get('propertyType'),
        'extendedPropertyType': detalhes.get('subTypology') or detalhes.get('typology'),
        'constructedArea': imovel_json.get('size'),
        'usableArea': None,
        'plotOfLand': None,
        'roomNumber': imovel_json.get('rooms'),
        'bathNumber': imovel_json.get('bathrooms'),

        'isInTopFloor': None,
        'isDuplex': 'duplex' in imovel_json.get('propertyType', '').lower(),
        'flatLocation': None,
        'isStudio': 'studio' in imovel_json.get('propertyType', '').lower(),
        'agencyIsABankisPenthouse': None,
        'energyCertificationType': None,
        'energyPerformance': None,
        'status': imovel_json.get('status'),
        'lift': imovel_json.get('hasLift'),
        'isAuction': None,
        'boxroom': None,
        'swimmingPool': imovel_json.get('hasSwimmingPool'),
        'chimney': None,
        'garden': None,
        'communityCosts': None,
        'heaterType1': None,
        'heaterType2': None,
        'construction_year': None,
        'manyFloors': None,
        'orientation': None,
        'terrace': None,

        'has_garage': estacionamento.get('hasParkingSpace', False),
        'garage_price': None,

        'floorNumber': imovel_json.get('floor'),
        'aircondicioning': imovel_json.get('hasAirConditioning')
    }

    return dados_padronizados

In [ ]:
import pandas as pd

df_raspado = pd.read_csv('base_cordoba_api.csv')
df_gabarito = pd.read_csv('foo.csv', nrows=0)

mapa_colunas = {
    'ubicacion_latitude': 'ubication__latitude',
    'ubicacion_longitud': 'ubication__longitude',
    'ubicacion_HasHidden': 'ubication__hasHiddenAddress',
    'has_garage': 'has garage',
    'garage_price': 'garage price',
    'construction_year': 'construction year',
    'aircondicioning': 'air conditioning'
}

df_raspado = df_raspado.rename(columns=mapa_colunas)
df_final = df_raspado.reindex(columns=df_gabarito.columns)

print(f"Sucesso! A sua base raspada agora tem {df_final.shape[1]} colunas, alinhada com el modelo.")

df_final.to_csv('base_idealista_cordoba_modelo.csv', index=False, encoding='utf-8-sig')
display(df_final.head(3))

Sucesso! A sua base raspada agora tem 47 colunas, alinhada com el modelo.


,id,url,operation,state,location_id,ubication__latitude,ubication__longitude,ubication__hasHiddenAddress,ubication__administrativeAreaLevel1,ubication__administrativeAreaLevel2,...,heaterType1,heaterType2,construction year,manyFloors,orientation,terrace,has garage,garage price,floorNumber,air conditioning
0,111804550,https://www.idealista.com/inmueble/111804550/,sale,Córdoba,Córdoba,37.924415,-4.790413,True,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,True,NaN,NaN,NaN
1,109756504,https://www.idealista.com/obra-nueva/109756504/,sale,Córdoba,Córdoba,37.868662,-4.794426,False,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,True,NaN,3,NaN
2,109100668,https://www.idealista.com/inmueble/109100668/,sale,Córdoba,Córdoba,37.885747,-4.780557,True,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,True,NaN,2,NaN


In [ ]:
from google.colab import files
files.download('base_idealista_cordoba_modelo.csv')

In [2]:
import pandas as pd

# Simulando los DataFrames que vendrían de tus archivos CSV
df_ayer = pd.DataFrame({'id': [101, 102, 103, 104], 'barrio': ['Centro', 'Sur', 'Norte', 'Este']})
df_hoy = pd.DataFrame({'id': [102, 103, 104, 105, 106], 'barrio': ['Sur', 'Norte', 'Este', 'Oeste', 'Centro']})

# Calculando la diferencia de conjuntos
nuevos_ids = set(df_hoy['id']) - set(df_ayer['id'])

print("IDs nuevos encontrados:", nuevos_ids)

IDs nuevos encontrados: {105, 106}


In [3]:
# Filtramos el DataFrame para extraer las filas completas de los anuncios nuevos
anuncios_nuevos = df_hoy[df_hoy['id'].isin(nuevos_ids)]

print("\nLos datos completos de los anuncios nuevos son:")
print(anuncios_nuevos)


Los datos completos de los anuncios nuevos son:
    id  barrio
3  105   Oeste
4  106  Centro


In [10]:
from google.colab import files
import pandas as pd
import io

print("Por favor, sube el archivo con los datos de AYER (ej: base_cordoba_api.csv):")
uploaded = files.upload()

# Obtenemos el nombre exacto del archivo que acabas de subir
nombre_archivo = list(uploaded.keys())[0]

# Leemos el archivo histórico
df_ayer = pd.read_csv(io.BytesIO(uploaded[nombre_archivo]))

print(f"\n¡Éxito! Archivo histórico '{nombre_archivo}' cargado en memoria.")

# Comparamos usando la lógica de conjuntos que ya dominas
nuevos_ids = set(df_hoy['id']) - set(df_ayer['id'])

# Filtramos para obtener las filas completas de los anuncios nuevos
anuncios_nuevos = df_hoy[df_hoy['id'].isin(nuevos_ids)]

print(f"\nSe han detectado {len(anuncios_nuevos)} anuncios nuevos en Córdoba.")
print(anuncios_nuevos.head()) # Muestra un resumen de los primeros resultados

Por favor, sube el archivo con los datos de AYER (ej: base_cordoba_api.csv):


Saving base_cordoba_api.csv to base_cordoba_api.csv

¡Éxito! Archivo histórico 'base_cordoba_api.csv' cargado en memoria.

Se han detectado 5 anuncios nuevos en Córdoba.
    id  barrio
0  102     Sur
1  103   Norte
2  104    Este
3  105   Oeste
4  106  Centro


In [11]:
from datetime import datetime

# 1. Capturamos la fecha actual en formato AÑO-MES-DIA
fecha_hoy = datetime.now().strftime('%Y-%m-%d')

# 2. Insertamos la fecha en la ruta del archivo usando una f-string
ruta_salida = f'data/anuncios_nuevos_{fecha_hoy}.csv'

# 3. Exportamos los datos limpios
anuncios_nuevos.to_csv(ruta_salida, index=False, encoding='utf-8-sig')

print(f"¡Guardado con éxito: {ruta_salida}!")

OSError: Cannot save file into a non-existent directory: 'data'